In [1]:
import numpy as np
import pandas as pd
import pickle
import json
import logging
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score, recall_score, roc_auc_score, f1_score
import string
import mlflow
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

c:\Users\rajes\Desktop\mlops\mlops-mini-project\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv").drop(columns=['tweet_id'])

In [3]:
df

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...
...,...,...
39995,neutral,@JohnLloydTaylor
39996,love,Happy Mothers Day All my love
39997,love,Happy Mother's Day to all the mommies out ther...
39998,happiness,@niariley WASSUP BEAUTIFUL!!! FOLLOW ME!! PEE...


In [4]:
# Transform the data

import re
import pandas as pd
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download required NLTK resources
nltk.download("stopwords")
nltk.download("wordnet")


def lemmatization(text: str) -> str:
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)


def remove_stop_words(text: str) -> str:
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)


def removing_numbers(text: str) -> str:
    text = " ".join(
        [word for word in text.split() if not word.isdigit()]
    )
    return text


def lower_case(text: str) -> str:
    return text.lower()


def removing_punctuation(text: str) -> str:
    text = re.sub(
        '[%s]' % re.escape(r"""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""),
        " ",
        text
    )
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def removing_urls(text: str) -> str:
    url_pattern = re.compile(r"https?://\S+|www\.\S+")
    return url_pattern.sub("", text)


def normalize_text(df: pd.DataFrame) -> pd.DataFrame:
    df["content"] = df["content"].astype(str)

    df["content"] = df["content"].apply(lower_case)
    df["content"] = df["content"].apply(removing_urls)
    df["content"] = df["content"].apply(removing_numbers)
    df["content"] = df["content"].apply(removing_punctuation)
    df["content"] = df["content"].apply(remove_stop_words)
    df["content"] = df["content"].apply(lemmatization)

    return df


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rajes\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\rajes\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [5]:
df = normalize_text(df)
df.head()

,sentiment,content
0,empty,tiffanylue know listenin bad habit earlier sta...
1,sadness,layin n bed headache ughhhh waitin call
2,sadness,funeral ceremony gloomy friday
3,enthusiasm,want hang friend soon
4,neutral,dannycastillo want trade someone houston ticke...


In [6]:
df['sentiment'].value_counts()

sentiment
neutral       8638
worry         8459
happiness     5209
sadness       5165
love          3842
surprise      2187
fun           1776
relief        1526
hate          1323
empty          827
enthusiasm     759
boredom        179
anger          110
Name: count, dtype: int64

In [7]:
x = df['sentiment'].isin(['happiness','sadness'])
df = df[x]

In [8]:
df['sentiment'] = df['sentiment'].replace({'sadness':0, 'happiness':1})

C:\Users\rajes\AppData\Local\Temp\ipykernel_13588\1508851655.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'] = df['sentiment'].replace({'sadness':0, 'happiness':1})
C:\Users\rajes\AppData\Local\Temp\ipykernel_13588\1508851655.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sentiment'] = df['sentiment'].replace({'sadness':0, 'happiness':1})


In [9]:
df

,sentiment,content
1,0,layin n bed headache ughhhh waitin call
2,0,funeral ceremony gloomy friday
6,0,sleep im thinking old friend want married damn...
8,0,charviray charlene love miss
9,0,kelcouch sorry least friday
...,...,...
39986,1,going watch boy striped pj hope cry
39987,1,gave bike thorough wash degrease grease think ...
39988,1,amazing time last night mcfly incredible
39994,1,succesfully following tayla


In [10]:
vectorizer = CountVectorizer(max_features=1000)
X = vectorizer.fit_transform(df['content'])
y = df['sentiment']

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [13]:
import dagshub
import mlflow


dagshub.init(repo_owner='rajeshxdatascience', repo_name='mlops-mini-project', mlflow=True)
mlflow.set_tracking_uri("https://dagshub.com/rajeshxdatascience/mlops-mini-project.mlflow")

mlflow.set_experiment('Logistic Regression Baseline')

Accessing as rajeshxdatascience

Initialized MLflow to track repo "rajeshxdatascience/mlops-mini-project"

Repository rajeshxdatascience/mlops-mini-project initialized!

2026/07/12 17:35:55 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/bc1079e26ca3484386d808bcf7529072', creation_time=1783857957322, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1783857957322, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [14]:
with mlflow.start_run():

    # Log preprocessing parameters
    mlflow.log_param("vectorizer", "Bag of Words")
    mlflow.log_param('num_features', 1000)
    mlflow.log_param("test_size", 0.2)

    # Model building and training
    model = LogisticRegression()
    model.fit(X_train, y_train)

    mlflow.log_param("model", "Logistic Regression")

    # Model Evaluation
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    # Log evaluation matric
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("reccall", recall)
    mlflow.log_metric("f1", f1)

    # Log model
    mlflow.sklearn.log_model(model, "model")

    # save and log the notebook
    import os
    notebook_path = 'exp1_baseline_model.ipynb'
    os.system(f"jupyter nbconvert --to notebook --execute --inplce {notebook_path}")
    mlflow.log_artifact(notebook_path)

    # print
    print(f"Accuracy: {accuracy}")
    print(f"precision: {precision}")
    print(f"recall: {recall}")
    print(f"f1: {f1}")

2026/07/12 17:45:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Accuracy: 0.7763855421686747
precision: 0.7687804878048781
recall: 0.7763546798029557
f1: 0.7725490196078432
🏃 View run polite-mare-739 at: https://dagshub.com/rajeshxdatascience/mlops-mini-project.mlflow/#/experiments/0/runs/8229febd399d497292ba5a5c7566d80f
🧪 View experiment at: https://dagshub.com/rajeshxdatascience/mlops-mini-project.mlflow/#/experiments/0
